In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time
import re
import csv

In [2]:
pd.set_option('display.max_colwidth', None)

In [9]:
RESULTS_DIR = "../../../results/"
DATA_FILE = "n80_examples_large_v01_gpt_v01_10K_b12_v01.csv"
BENCHMARK_DIR = "../semantic_categories/"
DB_PATH = ".........../v33_koondkorpus_transaktsioonid_v04_2.db"


* geo_loc: geograafilised kohad 
* object_loc: objektid, mis võivad olla kohad 
* org_loc: organisatsioonid, mis võivad olla kohad 
* event_loc: tegevused/sündmused, millel on korraga nii aja kui koha tähendus
* per_loc: inimene kui koht 
* abstract_loc: abstraktsed kohad, mille asukoht on kas ebamäärane või ei eksisteerigi

## Vastustega df

In [10]:
df = pd.read_csv(RESULTS_DIR+DATA_FILE, encoding="utf-8", sep=",")

In [12]:
saving_columns = ["sentence_id", "head_id", 
                "verb", "verb_compound", "morph_case", "lemma", "form", "sentence",
                "timex_tag", "ekilex_tag", "ner_tag"]

In [13]:
len(list(df["lemma"].unique()))

5081

In [1]:
#print(list(df["lemma"].unique()))

In [189]:
counts2 = df.groupby(["lemma"], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,lemma,count
506,Eesti,56
1399,Tallinn,52
2936,kodu,45
1064,Moskva,41
3051,kool,37
...,...,...
2016,ekraan,1
2017,eks,1
2019,eksamiruum,1
2020,ekskrement,1


In [ ]:
# algus -> ajamäärus? kordub 21 korda andmetes
# aeg -> ?? 13 korda

# mille alla kuuluks "lähedus"? "lõpp". tee lõpus, nädala lõpus

In [85]:
alive_file = "........./v05_wordlists_obl/alive.txt"


potential_per = ["kobakäpp","juut", "ema", "vanaema","vanaisa", "tädi", "onu", "õde","tema", 
                 "mina", "meie", "teie", "nemad", "kaitseliitlane", "isa", "õde", "vend", "arst", 
                 "maalane", "müürsepp", "klient", "kunstnik", "muusik", "Raul", "Ackermann", 
                 "naaber", "kaaslane", "kass", "koer", "loom", "lind", "mees", "naine", "laps"]


with open(alive_file, encoding="utf-8") as f:
    alive = f.readlines()
    
alive = [e.strip() for e in potential_per]

In [86]:
pot_alive = []

for elem in list(df["lemma"].unique()):
    if elem in alive:
        pot_alive.append(elem)
#pot_alive        

### abstract_loc
* ebamäärased suunad/teekonnad: ida, trajektoor, liikus ummikteel 
* nähtamatud/abstraktsed/määratlemata piirideta alad: Wifi, kvantmaailm, arvutiturg, õhuruum, digitaalplatvorm, liigub läheduses, hommikukaste, rambivalgus
* veebisaidid, telekanalid: Delfi, Yle
* abstraktsed mõisted: kirjanduses liiguvad väited, lahkusin poliitikast/võimult, kasutusaladel käib testimine
* ülekantud tähendusega füüsiline liikumine: istus hooaja jooksul peatreeneripingile - sai peatreeneriks, istus lavastajapuldis - oli lavastaja


In [70]:
potential_abs = ["lääs", "popmuusika","ETV", "hämar", "valgus","ringkond","valdkond","teadvus", 
                 "trajektoor","teekond", "rööbas",  "ummiktee", "internet", "kvantmaailm", "arvutiturg",
                 "õhuruum", "digitaalplatvorm", "läheduses", "hommikukaste", "rambivalgus", "Delfi", 
                 "kirjandus", "poliitikia", "võim", "kasutusala", "mõttemaailm", "teadvus",
                "kultuur", "maastik", "süsteem", "maailmavaade", "sektor", "sfäär", "vaatenurk",
                "ühiskond", "kogukond", "õigusruum"]

undes = ["raamatupida", "elamu", "maja", "juht", "server", "firma", "valgusti", "ajakirjandusväljaanne"]
abs1 = df[(df["lemma"].str.contains('|'.join(potential_abs))) | (df["lemma"].isin(["ida", "veeb", "elu", "küsimus"])) | (df["form"].str.contains('|'.join(["mõtetes"])))]
abs1 = abs1[~(abs1["lemma"].str.contains('|'.join(undes)))]
abs1 = abs1.sample(frac=1)

abs1 = abs1.drop_duplicates(subset="form")


abs1 = abs1.iloc[:100]
abs1 = abs1[saving_columns]
#abs1

In [71]:
len(abs1)

100

In [74]:
abs1[abs1["lemma"]=="sfäär"]

,sentence_id,head_id,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag
5256,2935745,4706863,ringlema,NaN,in,sfäär,sfääris,"Kuna olen naftat puurinud ja geoloogilistel ekspeditsioonidel käinud , kõik mu noorepõlve aastad on ju rännakute rõõmud , siis olen selles sfääris ringelnud .",NaN,NaN,NaN


In [72]:
abs1["lemma"].unique()

array(['kirjandus', 'internetiportaal', 'lääs', 'popmuusika',
       'alateadvus', 'erasfäär', 'valdkond', 'kultuuriõu', 'internetiilm',
       'eneseteadvus', 'ajakirjandus', 'kultuuripalee', 'kogukond',
       'valgus', 'ETV', 'ühiskond', 'infoühiskond', 'Delfi', 'elu',
       'eluvaldkond', 'mõte', 'atmosfäär', 'ajalookirjandus', 'teadvus',
       'kultuurikeskus', 'sfäär', 'kultuuriruum', 'ringkond', 'ida',
       'küsimus', 'süsteem', 'inimühiskond', 'IT-süsteem', 'mõõtsüsteem',
       'EDI-süsteem', 'haridussüsteem', 'kultuuriringkond', 'internet',
       'sektor', 'kultuurilooline', 'lastekirjandus', '10-palli-süsteem',
       'kultuur', 'finantssektor', 'kultuuritrummel', 'ametisfäär',
       'IT-valdkond', 'hämarus', 'fännisektor', 'mõjusfäär',
       'kultuurkapital', 'kodukultuuriring', 'erasektor',
       'meditsiinisüsteem', 'jäätmekäitlus-süsteem', 'ehitusvaldkond',
       'internetipood', 'Delfi-kari', 'külakogukond', 'kultuuripealinn',
       'ajakirjanduspilt', 'poolhä

In [75]:
abs1.to_csv(BENCHMARK_DIR+"abstract_locations/positive_set_01.csv", sep=",", encoding="utf-8", index=False, quoting=csv.QUOTE_MINIMAL)

### geo_loc
* kohanimed: Bristol, Sepphoris
* ehitised/äride füüsilised asukohad: pangamaja, multimeediastuudio, Kuku klubi, käisime arvutifirmas
* alad, mille geograafiline asukoht on defineeritav: põlengupaik, põhjapoolus, kaldapealne, tagaots, tolmupilv
* koju

In [66]:
potential_loc = ["maja", "kabinet", "stuudio", "koht", "klubi", "firma", "paik", "pealne", "poolus", 
                 "Aafrika", "Eesti", "Narva", "Tartu"]

undes = ["protsess", "esikoht","aukoht", "majandus", "pidamine", "rida", "mida", "kohtumine", "aeg", "majapidamine", "liidri", "võistlus", "ehitus", "Riigi", "majand", "kohtuinstants", "omanikfirma", "konverents", "kaalumaja"]
geo1 = df[(df["lemma"].str.contains('|'.join(potential_loc)))]
geo1 = geo1[~(geo1["lemma"].str.contains('|'.join(undes)))]
geo1 = geo1.sample(frac=1)

geo1 = geo1.drop_duplicates(subset="form")

geo1 = geo1.iloc[:100]
geo1 = geo1[saving_columns]
#geo1

In [24]:
len(geo1["form"].unique())

100

In [26]:
geo1.to_csv(BENCHMARK_DIR+"geographic_locations/positive_set_01.csv", sep=",", encoding="utf-8", index=False, quoting=csv.QUOTE_MINIMAL)

### per_loc

In [87]:

undes = ["kabinet","planeering","hoidlane","kleit",  "juhtme","sild", "olematu","süvend", "Iisaku", "teema", "saal", "mõle", "keel", "ala", "korter", "saar" , "kool", "tund", "klass", "ruum", "hoone", "muusika", "sarnane", "kinnis", "eelarve", "kodu", "paik", "maa", "kond", "Kenema", "nimekiri", "talu", "sadam", "riik"]
per1 = df[(df["lemma"].isin(potential_per)) | (df["lemma"].str.contains('|'.join(["maalane", "kaaslane", "kapten", "lane"])))]
per1 = per1[~(per1["lemma"].str.contains('|'.join(undes)))]
per1 = per1.sample(frac=1)

per1 = per1.drop_duplicates(subset="form")

per1 = per1.iloc[:100]
per1 = per1[saving_columns]
per1

,sentence_id,head_id,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag
960,3363170,5410857,röövima,NaN,el,prantslane,prantslasest,"DUBAI , 31. august ( Reuters-EPLO ) - Iraagis kaks prantslasest ajakirjanikku röövinud kurjategijad andsid Prantsusmaale veel 24 tundi aega tühistamaks moslemite pearättide kandmise keeld riigikoolides .",NaN,NaN,NaN
8704,3222616,5176701,vahendama,NaN,ill,tema,Neisse,Neisse vahendab kohti korralduskomiteega lepingu sõlminud turismibüroo .,NaN,NaN,NaN
7982,821266,1310149,röövima,NaN,el,leedulane,leedulasest,"61-aastaselt leedulasest bussijuhilt elu röövinud õnnetus toimus Eurolines Eesti juhi Hugo Osula sõnul eile kella kahe ajal pärastlõunal kohaliku aja järgi Salacgriva asula läheduses , 14-15 kilomeetri kaugusel Eesti piirist .",NaN,alive,NaN
8991,12236892,19590430,astuma,sisse,ill,kapten,Kaptenisse,"Tea , kas olekski üldse Kaptenisse sisse astunud , kui väljas oleks sadanud ja väliterrassil poleks saanud istuda , sest ei oska arvata , mis nipiga õnnestuks ilusal soojal suvepäeval kõrtsi suurde , ilmselgelt väsinud ja kulunud saali meelitada .",NaN,alive,NaN
7241,17652190,26987770,nägema,NaN,ill,mina,minusse,"Ehkki mõni võib ette kujutada et « näe minusse suhtuti hästi , seega suhtuvad ka 5,5 miljardit inimest Eestisse hästi » aga tegelikkuses teavad eestlastest midagi ehk naaberriikides elavad inimesed , end .",NaN,NaN,NaN
246,12424987,19894347,külastama,NaN,adit,müürsepp,Müürseppa,Veebruaris Müürseppa külastanud endine treener Andres Sõber peab sellist meediakära ülepaisutatuks .,NaN,alive,NaN
6577,7501494,12042946,voolama,välja,el,ema,emast,"Nad jooksevad mitu tiiru ümber turuplatsi , nende tee viib kiriku ja selle ees bussiootepingil oigava ema eest läbi ; emast voolavad sünnitusveed välja , pingi ees on juba väike lomp , Tiigrile kukkumiseks , libisemiseks ja siis mahaprantsatamiseks - sel juhul oleks minagi justkui võitlusest osa võtnud , võitu oma panuse andnud .",NaN,NaN,NaN
8940,19992621,29393515,kõndima,NaN,in,mina,mus,laaneke: mudu vaata pühapäeval sitarada et mus mileedi kõndis,NaN,NaN,NaN
2094,5786334,9284013,liikuma,NaN,in,Raul,Raulis,"Raulis liigub kindlalt , katuseharjal kingakontsade abil tasakaalu hoides .",NaN,NaN,NaN
6773,2193416,3505863,helisema,NaN,in,tema,temas,"Debora on ennekõike muusik , temas helisevad meloodiad ja tormlevad rahutud , ent nõtked rütmid , mängumaneer on elegantne ja tehnika nauditav , tema viiulikeeltelt kerkib publiku poole inimlikku soojust .",NaN,NaN,NaN


In [88]:
len(per1)

37

In [35]:
per1["lemma"].unique()

array(['tema', 'prantslane', 'kaitseliitlane', 'eestlane', 'Raul', 'mina',
       'vend', 'kapten', 'arst', 'juut', 'müürsepp', 'õde', 'kobakäpp',
       'leedulane', 'Ackermann', 'ema', 'albaanlane', 'kunstnik',
       'klient'], dtype=object)

In [89]:
per1.to_csv(BENCHMARK_DIR+"living_entities/positive_set_01.csv", sep=",", encoding="utf-8", index=False, quoting=csv.QUOTE_MINIMAL)

### event_loc

* kleidiproov, värbamine, haldusmenetlus, prostitutsiooniprotsess, suusatreening jne

In [37]:
events_file = "......../v05_wordlists_obl/event.txt"

with open(events_file, encoding="utf-8") as f:
    potential_events = f.readlines()
    
potential_events = [e.strip() for e in potential_events]

In [41]:
searchfor = ["proov", "värbamine", "menetlus", "protsess", "treening", "koosolek", 
             "pidu", "rünnak", "sõit", "pulm", "sõda", "etendus", "peied", 
             "laat", "näitus", "matk", "õnnetus", "reis", "teenistus", "jaht", 
             "seik", "kogunemine", "hakkamine", "võtmine", "kontsert", "seminar",
            "etendus", "festival", "kogunemine", "loeng", "näitus", "konkurss", 
            "mess", "kokkutulek", "maraton", "piknik", "lõpetamine"]

potential_events += searchfor
potential_events = list(set(potential_events))

undes = ["ruum", "maja", "konteiner", "saal", "teine", "karp", "piletisaba", "keskus", "Georgia", "toimetulek", "REMOTEHOST", "püks", "plaat", "paik", "komitee", "viljant", "POMM", "asutus", "tabel", "reklaam", "söökla", "klass", "Kapellskär", "poliitika", "liiga", "folkloor", "play", "EMEX", "linn", "koht", "võistlustuli", "park", "EMU", "raamat", "Ljantor", "EMOR", "kett", "kõnts", "võistlusala", "joove", "kava" ]

ev1 = df[(df["lemma"].str.contains('|'.join(potential_events))) &  ~(df["lemma"].str.contains('|'.join(undes)))]

ev1 = ev1.sample(frac=1)

ev1 = ev1.drop_duplicates(subset="form")

ev1 = ev1.iloc[:100]
ev1 = ev1[saving_columns]
#ev1

In [42]:
len(ev1["form"].unique())

100

In [43]:
ev1.to_csv(BENCHMARK_DIR+"events/positive_set_01.csv", sep=",", encoding="utf-8", index=False, quoting=csv.QUOTE_MINIMAL)

### org_loc
* organisatsioonid/kollektiivid: istun valitsuses, käin ülikoolis, hokitrennis, liigun võrgustikesse, lahkun töökohalt, vormelimaailm (koolid, trennid, lasteaiad)

In [55]:
searchfor = ["trenn", "laager", "valitsus", "töökoht", "kool", "liit", "nõukogu", 
             "lasteaed", "klubi", "muuseum", "raamatukogu", "teater", "ühing", "ansambel",
            "firma", "haigla", "galerii"]

undes = ["koolkond", "koolimaja", "koolikoridor", "koolihoone", "koolihoov", "koolisöökla", "keskkooliaste", "otsa-kool", "ülikoolilinn", "valitsuskvartal", "monoliitsus", "poliitik", "eliit"]
org1 = df[(df["lemma"].str.contains('|'.join(searchfor))) &  ~(df["lemma"].str.contains('|'.join(undes)))]
org1 = org1.sample(frac=1)

org1 = org1.drop_duplicates(subset="form")

org1 = org1.iloc[:100]
org1 = org1[saving_columns]
org1

,sentence_id,head_id,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag
2589,2432622,3900819,ootama,NaN,ill,lasteaed,lasteaedadesse,""" Praegune hind käib ikka paljudele üle jõu , samal ajal ootab näiteks Nõmmel linna lasteaedadesse järjekorras 700 last , "" nentis ta .",NaN,location,NaN
7653,12703397,20323503,järgnema,NaN,ill,laager,laagrisse,"Filmi läbiv teema , Guido ja Dora piiritu armastus , saab sügavama tähenduse , kui Guido talle vabatahtlikult laagrisse järgnenud naisele pisikeste asjade abil aegajalt meelde tuletab , et elu on ilus .",NaN,NaN,NaN
6683,1978277,3155001,säilima,NaN,in,firma,firmas,"Tänaseks on teada , et ühinenud firmas säilivad mõlemad kaubamärgid - üldehitus ja kinnisvara arendus hakkab toimuma Merko nime all , inseneriehitus ja ehitusmaterjalitööstus aga EMV sildi all .",NaN,NaN,NaN
280,1568144,2495412,suunama,NaN,ill,firma,firmasse,"Me vastame küsimusele , soovitame temaatilist kirjandust oma infoletist ning vajadusel suuname firmasse , kust ta saab vajaliku seadme , materjali või teenuse osta , "" selgitas ta .",NaN,NaN,NaN
6472,11881617,19012323,astuma,läbi,el,õmblusfirma,õmblusfirmast,"Seal väisab presidendipaar presidendikantselei teatel Albu põhikooli kunstiajalootundi , kohtub vallarahvaga , astub läbi Ahula õmblusfirmast Pakpoord ning käib Anton Hansen Tammsaare muuseumis Vargamäel .",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1897,3528551,5682340,algama,NaN,ill,jahtklubi,jahtklubisse,ETA teatel algavad merepäevade üritused täna Haapsalus kell 17 linna puhkpilliorkestri marsiga kaubanduskeskusest jahtklubisse .,NaN,location,NaN
6220,9049955,14551454,võtma,kaasa,el,klubi,klubist,"Kirsipuu ja Aus võtsid Casino klubist kaasa massööri , kes hoolitseb kõigi Eesti koondislaste eest .",NaN,NaN,NaN
2446,2586971,4151894,jääma,maha,ill,galerii,galeriisse,"Jaanisoo raiub aga üheksakümnendate aastate soojaga edasi , temast jäävad galeriisse maha tüüakad ja kobedad objektid , millele hea käsi peale panna ja mis viskavad pikka ümarat varju .",NaN,NaN,NaN
6404,12467866,19958882,sadama,maha,in,laager,laagris,"Eesti meessuusatajate laagris Põhja-Soomes Oloksel sadas ööl vastu kolmapäeva maha paras lumevaip , mis lisas vägevalt treeninguindu .",NaN,NaN,NaN


In [56]:
len(org1["form"].unique())

100

In [59]:
org1.to_csv(BENCHMARK_DIR+"organisations/positive_set_01.csv", sep=",", encoding="utf-8", index=False, quoting=csv.QUOTE_MINIMAL)

### object_loc

* füüsilised objektid: esikohapoodium, Kuu, varundusseade, sadul, pilv, põuetasku
* kui väljendatakse abstraktset nähtust, aga objekti geograafiline asukoht on ikka määratav (nt hirm käib luust läbi - tegelikult pole hirm luus, aga luu on ise ikkagi kindla asukohaga)


In [63]:
searchfor = ["seade", "poodium", "mänguasi", "pirukas", "tasku", "pilv", "karp", 
             "masin", "lennuk", "käru", "vitriin", "sadul", "putka", "auto", "katel",
             "kott", "ketas", "päevik", "kasukas", "medal", "uks", "ratas", "laev", "rong",
            "kohver", "kast", "ämber", "korv", "pall"]
undes = ["turg", "bussitasku", "autoinspektsioon", "vang", "masingam", "pesula", "Luksemburg", "Aluksne", "medalikolmik", "tehas", "salong", "äri", "firma", "keskus", "pood", "baas", "parkla", "hotell", "automaat", "autorikaitse", "avarii", "register", "teenindus", "autotee", "võidusõit", "esikuuks"]
obj1 = df[(df["lemma"].str.contains('|'.join(searchfor))) &  ~(df["lemma"].str.contains('|'.join(undes))) | df["form"].str.contains("silmadest") | df["form"].str.contains("näkku")]
obj1 = obj1.sample(frac=1)

obj1 = obj1.drop_duplicates(subset="form")

obj1 = obj1.iloc[:100]
obj1 = obj1[saving_columns]
#obj1

In [64]:
len(obj1["form"].unique())

100

In [65]:
obj1.to_csv(BENCHMARK_DIR+"physical_objects/positive_set_01.csv", sep=",", encoding="utf-8", index=False, quoting=csv.QUOTE_MINIMAL)

## Lisada phrase_root_loc benchmark failidele

In [76]:
import sqlite3
import pandas as pd
import os, sys, re
import csv

In [93]:
files = [
    "../semantic_categories/abstract_locations/positive_set_01.csv",
    "../semantic_categories/events/positive_set_01.csv",
    "../semantic_categories/geographic_locations/positive_set_01.csv",
    "../semantic_categories/living_entities/positive_set_01.csv",
    "../semantic_categories/organisations/positive_set_01.csv",
    "../semantic_categories/physical_objects/positive_set_01.csv",
]

In [91]:
# database file path
filename = DB_PATH

# connecting with database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

In [94]:
# benchmark tabel andmebaasi

table_names = []

for fname in files:
    df1 = pd.read_csv(fname)
    table_name = "_".join(fname.split("/")[-2:]).replace(".csv", "")
    table_names.append(table_name)
    df1.to_sql(table_name, conn, if_exists="replace", index=False)

In [95]:
table_names

['abstract_locations_positive_set_01',
 'events_positive_set_01',
 'geographic_locations_positive_set_01',
 'living_entities_positive_set_01',
 'organisations_positive_set_01',
 'physical_objects_positive_set_01']

In [21]:
#for name in table_names:
#    cursor.execute(f"DROP TABLE if exists '{name}'")

In [96]:
# benchmark tabeli join transaction_row tabeliga et saada peasõna loc

for fname, tname in zip(files, table_names):


    query = f"""SELECT sentence_id, {tname}.head_id, transaction_row.loc as head_loc, 
                verb, verb_compound, morph_case,{tname}.lemma, {tname}.form, sentence,
                timex_tag, ekilex_tag, ner_tag
                FROM {tname}
                join transaction_row
                on 
                transaction_row.head_id={tname}.head_id and 
                transaction_row.form={tname}.form and 
                transaction_row.lemma={tname}.lemma
                """

    res = pd.read_sql(query, conn)

    
    res.to_csv(fname, sep=",", encoding="utf-8", index=False, quoting=csv.QUOTE_MINIMAL)
    print(tname, "->>", fname)
    #break


abstract_locations_positive_set_01 ->> ../semantic_categories/abstract_locations/positive_set_01.csv
events_positive_set_01 ->> ../semantic_categories/events/positive_set_01.csv
geographic_locations_positive_set_01 ->> ../semantic_categories/geographic_locations/positive_set_01.csv
living_entities_positive_set_01 ->> ../semantic_categories/living_entities/positive_set_01.csv
organisations_positive_set_01 ->> ../semantic_categories/organisations/positive_set_01.csv
physical_objects_positive_set_01 ->> ../semantic_categories/physical_objects/positive_set_01.csv


In [97]:
conn.close()